In [10]:
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, IntegerType


StatementMeta(, cbcd2a02-10d2-4c4f-a352-242028f2467d, 12, Finished, Available, Finished, False)

In [7]:
df_silver=spark.read.table("bronze_flipkart_products")
print(f"Records from Bronze: {df_silver.count()}")

# Clean up column names — remove spaces

df_silver=df_silver\
        .withColumnRenamed("uniq_id", "product_id")\
        .withColumnRenamed("product_name", "product_name")\
        .withColumnRenamed("product_category_tree", "category_tree")\
        .withColumnRenamed("retail_price", "retail_price")\
        .withColumnRenamed("discounted_price", "discounted_price")\
        .withColumnRenamed("description", "description")\
        .withColumnRenamed("product_rating", "product_rating")\
        .withColumnRenamed("overall_rating", "overall_rating")\
        .withColumnRenamed("brand", "brand")


print("Column names cleaned")


StatementMeta(, cbcd2a02-10d2-4c4f-a352-242028f2467d, 9, Finished, Available, Finished, False)

Records from Bronze: 20000
Column names cleaned


In [12]:
# Count before cleaning
before = df_silver.count()
print(f"Before cleaning: {before} products")

# Fix price columns — convert to numbers
df_silver = df_silver \
    .withColumn("retail_price",
        F.regexp_replace(
            F.col("retail_price"), "[^0-9.]", ""
        ).cast(DoubleType())) \
    .withColumn("discounted_price",
        F.regexp_replace(
            F.col("discounted_price"), "[^0-9.]", ""
        ).cast(DoubleType()))

# Remove rows with no product name
df_silver = df_silver.filter(
    F.col("product_name").isNotNull()
)

# Remove rows with no price
df_silver = df_silver.filter(
    F.col("retail_price").isNotNull() &
    F.col("discounted_price").isNotNull()
)

# Remove rows where discounted price
# is MORE than retail price (bad data)
df_silver = df_silver.filter(
    F.col("discounted_price") <= F.col("retail_price")
)

# Remove rows with zero or negative prices
df_silver = df_silver.filter(
    (F.col("retail_price") > 0) &
    (F.col("discounted_price") > 0)
)

after = df_silver.count()
print(f"After cleaning: {after} products")
print(f"Removed: {before - after} bad records")
print(f"Data quality: {round(after/before*100,1)}%")

StatementMeta(, cbcd2a02-10d2-4c4f-a352-242028f2467d, 14, Finished, Available, Finished, False)

Before cleaning: 19922 products
After cleaning: 19922 products
Removed: 0 bad records
Data quality: 100.0%


In [13]:
# Add columns that will be useful for analytics


df_silver=df_silver\
    .withColumn(
        "discount_amount",
        F.round(
            F.col("retail_price")-F.col("discounted_price"),
            2
        )
    )\
    .withColumn(
        "discount_percentage",
        F.round(
            (F.col("retail_price") - F.col("discounted_price"))
            / F.col("retail_price") * 100,
            1
        )
    ) \
    .withColumn(
        "price_category",
        F.when(F.col("discounted_price") < 500, "Budget")
         .when(F.col("discounted_price") < 2000, "Mid-Range")
         .when(F.col("discounted_price") < 10000, "Premium")
         .otherwise("Luxury")
    ) \
    .withColumn(
        "description_length",
        F.length(F.col("description"))
    ) \
    .withColumn(
        "main_category",
        F.trim(
            F.split(
                F.regexp_replace(
                    F.col("category_tree"),
                    '[\\[\\]"]', ""
                ),
                ">>"
            ).getItem(0)
        )
    )

print("Business columns added:")
print("  - discount_amount")
print("  - discount_percentage")
print("  - price_category (Budget/Mid-Range/Premium/Luxury)")
print("  - description_length")
print("  - main_category")

StatementMeta(, cbcd2a02-10d2-4c4f-a352-242028f2467d, 15, Finished, Available, Finished, False)

Business columns added:
  - discount_amount
  - discount_percentage
  - price_category (Budget/Mid-Range/Premium/Luxury)
  - description_length
  - main_category


In [14]:
# Save as Silver Delta table

df_silver.write\
    .format("delta")\
    .mode("overwrite")\
    .saveAsTable("silver_flipkart_products")


print("Silver layer complete")



# Quick check
print("\nPrice categories distribution:")
spark.sql("""
    SELECT price_category, COUNT(*) as products
    FROM silver_flipkart_products
    GROUP BY price_category
    ORDER BY products DESC
""").show()


print("\nTop 10 main categories:")
spark.sql("""
    SELECT main_category, COUNT(*) as products
    FROM silver_flipkart_products
    GROUP BY main_category
    ORDER BY products DESC
    LIMIT 10
""").show()

StatementMeta(, cbcd2a02-10d2-4c4f-a352-242028f2467d, 16, Finished, Available, Finished, False)

Silver layer complete

Price categories distribution:
+--------------+--------+
|price_category|products|
+--------------+--------+
|        Budget|    9385|
|     Mid-Range|    8633|
|       Premium|    1079|
|        Luxury|     825|
+--------------+--------+


Top 10 main categories:
+--------------------+--------+
|       main_category|products|
+--------------------+--------+
|            Clothing|    6171|
|           Jewellery|    3522|
|            Footwear|    1225|
|Mobiles & Accesso...|    1097|
|          Automotive|    1010|
|Home Decor & Fest...|     927|
|Beauty and Person...|     709|
|     Home Furnishing|     700|
|    Kitchen & Dining|     645|
|           Computers|     573|
+--------------------+--------+

